In [1]:
import pandas as pd
from bertopic import BERTopic

/home/webmedia2025/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_path = "best_mean/16/bertopic_model"
model = BERTopic.load(model_path)

df = pd.read_csv("processed_df.csv", engine="python")
sample_df = pd.read_csv("sample_df.csv", engine="python")

print(len(df))
print(len(sample_df))

673587
67359


In [3]:
sample_docs = sample_df['clean_text'].tolist()


In [4]:
model = BERTopic.load("best_mean/16/bertopic_model")
topics = model.get_topics()
docs_topics, probs = model.transform(sample_docs)


In [5]:
from IPython.display import display

def show_results(model):

    info = model.get_topic_info()
    display(info)

    topicos = [t for t in model.get_topics().keys() if t != -1]
    fig = model.visualize_barchart(topics=topicos, n_words=10)
    fig.show()

In [6]:
show_results(model)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,19150,-1_bgmi_ttaakaa_ferguson_rubin,"[bgmi, ttaakaa, ferguson, rubin, itrustcapital...",[ripple ripple case stop open caleb brown acco...
1,0,10227,0_meditation_bible_tarot_spiritual,"[meditation, bible, tarot, spiritual, frequenc...",[july sunday energy tarot timestampe sassyscor...
2,1,8665,1_lyricist_composer_lyric_guitar,"[lyricist, composer, lyric, guitar, album, voc...",[hope feat walker smith bethel walker smith co...
3,2,4057,2_newsmax_trump_cable_hannity,"[newsmax, trump, cable, hannity, forbe, gutfel...",[trump court biden campaign host discuss presi...
4,3,3432,3_airdrop_crypto_wallet_altcoin,"[airdrop, crypto, wallet, altcoin, trading, cr...",[hana testnet payment hana airdrop theke sbaai...
...,...,...,...,...,...
57,56,120,56_harts_lowrer_mamowli_tesowt,"[harts, lowrer, mamowli, tesowt, azrowyts, tog...",[owghigh havak oskeparowm hetewe news sots ial...
58,57,117,57_redact_natali_clayton_morris,"[redact, natali, clayton, morris, lear, natali...",[safe americans agent warn redact clayton morr...
59,58,112,58_mahyar_tousi_confidentially_patriotic,"[mahyar, tousi, confidentially, patriotic, mer...",[britain islamic member tousi exclusive suppor...
60,59,108,59_diddy_comb_crime_sean,"[diddy, comb, crime, sean, sidebar, mogul, cas...",[diddy gate ashton kutcher nervous celebrity s...


In [7]:
import pandas as pd
import random

# Função para pegar documentos mais representativos
def get_representative_docs(model, docs, topic_id, top_n=3):
    doc_info = model.get_document_info(docs)
    # Filtra pelos documentos do tópico
    topic_docs = doc_info[doc_info['Topic'] == topic_id]
    # Ordena pela probabilidade
    topic_docs = topic_docs.sort_values('Probability', ascending=False)
    # Retorna os top N documentos
    return topic_docs['Document'].tolist()[:top_n]

# Função para pegar documentos aleatórios de um tópico
def get_random_docs(model, docs, topic_id, n=10, seed=42):
    doc_info = model.get_document_info(docs)
    topic_docs = doc_info[doc_info['Topic'] == topic_id]['Document'].tolist()
    random.seed(seed)
    if len(topic_docs) <= n:
        return topic_docs
    return random.sample(topic_docs, n)

# Lista para armazenar os dados
data = []

for topic_id in model.get_topics().keys():
    if topic_id == -1:  # Ignorar tópico de ruído
        continue
    
    # Palavras mais representativas
    keywords = [word for word, _ in model.get_topic(topic_id)][:20]
    
    # Documentos mais representativos
    top_docs = get_representative_docs(model, sample_docs, topic_id, top_n=3)
    
    # 3 documentos aleatórios
    random_docs = get_random_docs(model, sample_docs, topic_id, n=10)

    # Adiciona ao dataset
    row = {
        "Topic_ID": topic_id,
        "Keywords": ", ".join(keywords),
        "Top_Doc_1": top_docs[0] if len(top_docs) > 0 else "",
        "Top_Doc_2": top_docs[1] if len(top_docs) > 1 else "",
        "Top_Doc_3": top_docs[2] if len(top_docs) > 2 else "",
    }
    
    # Adiciona os 10 documentos aleatórios
    for i in range(10):
        row[f"Rand_Doc_{i+1}"] = random_docs[i] if len(random_docs) > i else ""
    
    data.append(row)


# Cria DataFrame
df_topics = pd.DataFrame(data)

# Salva em CSV
df_topics.to_csv("regression/topics_representative_docs.csv", index=False, encoding="utf-8")

# Salva em TXT
with open("regression/topics_representative_random_docs.txt", "w", encoding="utf-8") as f:
    for _, row in df_topics.iterrows():
        f.write(f"--- Topic {row['Topic_ID']} ---\n")
        f.write(f"Keywords: {row['Keywords']}\n\n")
        f.write("Top Documents:\n")
        f.write(f"1. {row['Top_Doc_1']}\n")
        f.write(f"2. {row['Top_Doc_2']}\n")
        f.write(f"3. {row['Top_Doc_3']}\n\n")
        f.write("Random Documents:\n")
        for i in range(10):
            f.write(f"{i+1}. {row[f'Rand_Doc_{i+1}']}\n")
        f.write("\n" + "-"*80 + "\n\n")

print("Arquivos CSV e TXT salvos com sucesso!")


Arquivos CSV e TXT salvos com sucesso!


In [8]:
sample_df['topic'] = docs_topics

topic_interest = {
    -1: 0, 0: 1, 1: 1, 2: 1, 3: 0, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 
    11: 0, 12: 1, 13: 1, 14: 1, 15: 1, 16: 0, 17: 1, 18: 1, 19: 1, 20: 1, 
    21: 0, 22: 1, 23: 1, 24: 1, 25: 1, 26: 1, 27: 1, 28: 1, 29: 1, 30: 0, 
    31: 1, 32: 1, 33: 1, 34: 1, 35: 1, 36: 1, 37: 1, 38: 1, 39: 1, 40: 1, 
    41: 1, 42: 1, 43: 0, 44: 1, 45: 1, 46: 1, 47: 1, 48: 1, 49: 0, 50: 1, 
    51: 1, 52: 1, 53: 1, 54: 0, 55: 0, 56: 1, 57: 1, 58: 0, 59: 1, 60: 0
}

sample_df['interesse'] = sample_df['topic'].map(topic_interest)

display(sample_df)
sample_df.to_csv("regression/interest_df.csv")

,Unnamed: 0,video_id,title,description,title_description,lang,clean_text,topic,interesse
0,474588,BRhODqge9dY,Guns N' Roses - November Rain (Lyrics),Guns N' Roses - Use Your Illusion I[2 LP] - Do...,Guns N' Roses - November Rain (Lyrics) Guns N...,en,rose november rain lyric rose illusion double ...,1,1
1,410902,qmjXoaGxRQw,Traditional craftsmanship of the Mongol Ger an...,UNESCO: Representative List of the Intangible ...,Traditional craftsmanship of the Mongol Ger an...,en,traditional craftsmanship mongol associate cus...,-1,0
2,408471,zQRYk7stYIw,Angel Baby,A screen representation of Rosie and the Origi...,Angel Baby A screen representation of Rosie an...,en,angel baby screen representation rosie origina...,-1,0
3,541647,mhhMPI86jhQ,Climate activists block Dodge RAM TRX and face...,I hope you guys realize its fake :) pls dont d...,Climate activists block Dodge RAM TRX and face...,en,climate activist block dodge face consequence ...,-1,0
4,613477,IcD_22IC5l0,Ricky Gervais // Minorities // #shorts,NaN,Ricky Gervais // Minorities // #shorts,en,ricky gervais minority,25,1
...,...,...,...,...,...,...,...,...,...
67354,317933,WlejKQfnMZg,🎯✅IRAQI DINAR HOT MUST WATCH SHORTEST SWEETEST...,Thank you for watching! You GOT THIS! We can g...,🎯✅IRAQI DINAR HOT MUST WATCH SHORTEST SWEETEST...,en,iraqi shortest sweetest summary shake total bo...,26,1
67355,622734,WK4lKGNnVME,"Galatians 1:4 Who gave himself for our sins, t...","Galatians 1:1 Paul, an apostle, (not of men, n...","Galatians 1:4 Who gave himself for our sins, t...",en,galatians deliver present evil galatians paul ...,0,1
67356,73208,oBHpI-m6DEA,The Future of Cryptocurrency: Unleashing DeFi,"In this video, we delve into the exciting real...",The Future of Cryptocurrency: Unleashing DeFi ...,en,future cryptocurrency unleash defi delve excit...,3,0
67357,627638,RIuW-OJY2uY,Mark Farina- Mushroom Jazz vol. 1,1 Mr. Electric Triangle -- Bosha Nova 0:00\...,Mark Farina- Mushroom Jazz vol. 1 1 Mr. Elect...,en,mark farina mushroom jazz electric triangle bo...,1,1


In [9]:
from sentence_transformers import SentenceTransformer

embedding_model_name = "paraphrase-MiniLM-L6-v2"  # mesmo usado no BERTopic
embedding_model = SentenceTransformer(embedding_model_name, device='cuda')

docs = df['clean_text'].tolist()
embeddings = embedding_model.encode(docs, show_progress_bar=True, batch_size=64)


Batches: 100%|██████████| 10525/10525 [07:59<00:00, 21.94it/s] 


In [10]:
from umap import UMAP

umap_model = UMAP(
    n_neighbors=10,  # ajuste para o mesmo que no BERTopic
    n_components=10,  # ajuste conforme sua configuração
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

embeddings_reduced = umap_model.fit_transform(embeddings)

import numpy as np
np.save("regression/embeddings_reduced.npy", embeddings_reduced)
